In [1]:
!pip install -q git+https://github.com/facebookresearch/detectron2.git
# !pip install -U detectron2


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.8 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import os
import random
import numpy as np
import cv2
import json
import shutil
from PIL import Image
from tqdm import tqdm
from collections import defaultdict, Counter
import time
import copy
import warnings
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, hooks
from detectron2.data import DatasetMapper, build_detection_train_loader, build_detection_test_loader
from detectron2.data import detection_utils as utils
from detectron2.data.datasets.coco import load_coco_json
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.modeling import META_ARCH_REGISTRY
from detectron2.modeling.meta_arch import RetinaNet
from detectron2.modeling.meta_arch.retinanet import RetinaNetHead
from detectron2.modeling.backbone import BACKBONE_REGISTRY, Backbone, build_backbone
from detectron2.modeling.backbone.resnet import ResNet, ResNet as DetectronResNet, CNNBlockBase
from detectron2.modeling.backbone.fpn import FPN, LastLevelP6P7
from detectron2.layers import ShapeSpec, get_norm
from detectron2.data.transforms import RandomFlip, RandomRotation, RandomCrop, RandomContrast, RandomBrightness, RandomSaturation, AugInput
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.structures import BoxMode
from detectron2.utils.logger import log_every_n_seconds
from detectron2.utils.events import get_event_storage

warnings.filterwarnings("ignore")

# **data load/split**

In [3]:
# === CONFIG ===
annotations_folder = '/kaggle/input/cottonweeddet3/CottonWeedDet3/annotations'
images_folder = '/kaggle/input/cottonweeddet3/CottonWeedDet3/images'
output_folder = '/kaggle/working/coco_dataset'
output_images_folder = os.path.join(output_folder, "images")

# Create output folders
os.makedirs(output_folder, exist_ok=True)
os.makedirs(os.path.join(output_images_folder, "train"), exist_ok=True)  # ✅ NEW
os.makedirs(os.path.join(output_images_folder, "val"), exist_ok=True)    # ✅ NEW

# Classes
class_map = {
    'carpetweed': 0,
    'morningglory': 1,
    'palmer_amaranth': 2
}
categories = [{"id": v, "name": k, "supercategory": "weed"} for k, v in class_map.items()]

# Step 1: Collect all image-label data
image_data = []

for json_file in os.listdir(annotations_folder):
    if not json_file.endswith('.json'):
        continue

    with open(os.path.join(annotations_folder, json_file)) as f:
        data = json.load(f)

    for item_key in data:
        item = data[item_key]
        filename = item['filename']
        regions = item['regions']
        image_path = os.path.join(images_folder, filename)

        if not os.path.exists(image_path):
            print(f"[⚠️ Skipping] Image not found: {filename}")
            continue

        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"[❌ Error opening] {filename}: {e}")
            continue

        bboxes = []
        for region in regions:
            shape_attr = region['shape_attributes']
            region_attr = region['region_attributes']

            if shape_attr['name'] != 'rect':
                continue

            x = shape_attr['x']
            y = shape_attr['y']
            w = shape_attr['width']
            h = shape_attr['height']
            class_name = region_attr['class'].lower()

            if class_name not in class_map:
                continue

            bbox = {
                "category_id": class_map[class_name],
                "bbox": [x, y, w, h],
                "area": w * h
            }
            bboxes.append(bbox)

        if bboxes:
            image_data.append({
                "file_name": filename,
                "width": width,
                "height": height,
                "bboxes": bboxes
            })

# Step 2: Split into train and val
from sklearn.model_selection import train_test_split

# Extract a label for stratification (just first class per image for simplicity)
image_labels = []
for img in image_data:
    # You can also consider set of all class_ids in image if needed
    first_class_id = img['bboxes'][0]['category_id']
    image_labels.append(first_class_id)

# Stratified split
train_imgs, val_imgs = train_test_split(
    image_data, 
    test_size=0.2, 
    random_state=42, 
    stratify=image_labels
)

splits = {
    "train": train_imgs,
    "val": val_imgs
}

# Step 3: Write COCO-style JSONs and copy images
def create_coco_json(data_split, split_name):
    coco = {
        "images": [],
        "annotations": [],
        "categories": categories
    }

    ann_id = 1
    for img_id, item in enumerate(tqdm(data_split), start=1):
        new_img_path = os.path.join(output_images_folder, split_name, item["file_name"])
        old_img_path = os.path.join(images_folder, item["file_name"])

        # ✅ Copy image to appropriate split folder
        shutil.copyfile(old_img_path, new_img_path)

        coco["images"].append({
            "id": img_id,
            "file_name": f"{split_name}/{item['file_name']}",
            "width": item["width"],
            "height": item["height"]
        })

        for bbox in item["bboxes"]:
            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": bbox["category_id"],
                "bbox": bbox["bbox"],
                "area": bbox["area"],
                "iscrowd": 0
            })
            ann_id += 1

    # Save annotations
    json_path = os.path.join(output_folder, f"annotations_{split_name}.json")
    with open(json_path, 'w') as f:
        json.dump(coco, f, indent=4)
    print(f"✅ COCO annotations saved to: {json_path}")

# Generate both splits
create_coco_json(splits["train"], "train")
create_coco_json(splits["val"], "val")


100%|██████████| 678/678 [00:43<00:00, 15.59it/s]


✅ COCO annotations saved to: /kaggle/working/coco_dataset/annotations_train.json


100%|██████████| 170/170 [00:12<00:00, 14.13it/s]

✅ COCO annotations saved to: /kaggle/working/coco_dataset/annotations_val.json


In [4]:
# Path to your COCO annotation file
json_path = "/kaggle/working/coco_dataset/annotations_train.json"  # or val/test if needed

# Load the JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Create a mapping from category_id to category_name
category_id_to_name = {cat['id']: cat['name'] for cat in data['categories']}

# Count the number of annotations per class
class_counts = defaultdict(int)

for ann in data['annotations']:
    class_id = ann['category_id']
    class_counts[class_id] += 1

# Print results
print("Class distribution in train dataset:")
for class_id, count in class_counts.items():
    print(f"{category_id_to_name[class_id]}: {count} instances")


from collections import Counter
def count_classes(images):
    counter = Counter()
    for img in images:
        for bbox in img['bboxes']:
            counter[bbox['category_id']] += 1
    ordered_counts = {}
    for class_name, class_id in class_map.items():
        ordered_counts[class_name] = counter.get(class_id, 0)
    return ordered_counts


print("Train class distribution:", count_classes(splits["train"]))
print("Val class distribution:", count_classes(splits["val"]))



Class distribution in train dataset:
morningglory: 397 instances
carpetweed: 464 instances
palmer_amaranth: 359 instances
Train class distribution: {'carpetweed': 464, 'morningglory': 397, 'palmer_amaranth': 359}
Val class distribution: {'carpetweed': 138, 'morningglory': 89, 'palmer_amaranth': 85}


In [5]:
import torch
torch.cuda.empty_cache()


In [6]:
from detectron2.data.datasets.coco import load_coco_json
dataset_dicts = load_coco_json("/kaggle/working/coco_dataset/annotations_train.json", "/kaggle/working/coco_dataset/images/")
print(dataset_dicts[0]["annotations"][0])  # Should be a dict


{'iscrowd': 0, 'bbox': [1452.55, 63.26, 2729.9299999999994, 3936.74], 'category_id': 1, 'bbox_mode': <BoxMode.XYWH_ABS: 1>}


In [9]:
import torch
torch.cuda.empty_cache()

In [10]:
import torch
import time
from torch.cuda.amp import autocast, GradScaler
from detectron2.engine import DefaultTrainer
from detectron2.utils.logger import log_every_n_seconds
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances

class MixedPrecisionTrainer(DefaultTrainer):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.scaler = GradScaler() if cfg.SOLVER.AMP.ENABLED else None

    def run_step(self):
        """Run training step with mixed precision"""
        assert self.model.training, "Model must be in training mode"
        
        if not hasattr(self, '_data_loader_iter'):
            self._data_loader_iter = iter(self.data_loader)

        start = time.perf_counter()
        
        try:
            data = next(self._data_loader_iter)
        except StopIteration:
            self._data_loader_iter = iter(self.data_loader)
            data = next(self._data_loader_iter)
        
        data_time = time.perf_counter() - start
        
        with autocast(enabled=self.cfg.SOLVER.AMP.ENABLED):
            loss_dict = self.model(data)
            losses = sum(loss_dict.values())
        
        if self.scaler:
            self.scaler.scale(losses).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            losses.backward()
            self.optimizer.step()
        
        self.optimizer.zero_grad()
        
        metrics_dict = {k: v.detach().cpu().item() for k, v in loss_dict.items()}
        metrics_dict["data_time"] = data_time
        self.storage.put_scalars(**metrics_dict)

def register_datasets():
    for name in ["cottonweed_train", "cottonweed_val"]:
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)

    classes = ["carpetweed", "morningglory", "palmer_amaranth"]
    MetadataCatalog.get("cottonweed_train").set(thing_classes=classes)
    MetadataCatalog.get("cottonweed_val").set(thing_classes=classes)

    register_coco_instances(
        "cottonweed_train",
        {},
        "/kaggle/working/coco_dataset/annotations_train.json",
        "/kaggle/working/coco_dataset/images/"
    )
    register_coco_instances(
        "cottonweed_val",
        {},
        "/kaggle/working/coco_dataset/annotations_val.json",
        "/kaggle/working/coco_dataset/images/"
    )


# **large object**

In [20]:
import time
import torch
from torch.cuda.amp import autocast, GradScaler
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.utils.events import get_event_storage
from detectron2.data import build_detection_test_loader, DatasetCatalog, MetadataCatalog
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetMapper
from detectron2.structures import BoxMode
from detectron2 import utils
import numpy as np
import copy
import random
import cv2
from detectron2.data.transforms import RandomFlip, AugInput


# 1. Setup Config with Enhanced Large Object Detection
def setup_enhanced_model():
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_101_FPN_3x.yaml"))

    cfg.MODEL.DEVICE = "cuda"
    cfg.SOLVER.AMP.ENABLED = True

    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False

    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.GRADIENT_ACCUMULATION_STEPS = 8

    # Increased image sizes to better handle large objects
    cfg.INPUT.MIN_SIZE_TRAIN = (512, 640, 768)  # Larger image sizes
    cfg.INPUT.MAX_SIZE_TRAIN = 1024  # Increased from 768
    cfg.INPUT.MAX_SIZE_TEST = 768   # Increased from 512
    cfg.INPUT.RANDOM_FLIP = "horizontal"

    cfg.DATASETS.TRAIN = ("cottonweed_train",)
    cfg.DATASETS.TEST = ("cottonweed_val",)

    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_101_FPN_3x.yaml")
    cfg.MODEL.RETINANET.NUM_CLASSES = 3
    
    # Adjust detection parameters for better large object detection
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.01
    cfg.MODEL.RETINANET.TOPK_CANDIDATES_TEST = 1500  # Increased from 1000
    
    # Adjust focal loss parameters
    cfg.MODEL.RETINANET.FOCAL_LOSS_GAMMA = 1.5  # Decreased from 2.0
    cfg.MODEL.RETINANET.FOCAL_LOSS_ALPHA = 0.3  # Increased from 0.25
    cfg.MODEL.RETINANET.USE_CHECKPOINT = True

    cfg.MODEL.BACKBONE.FREEZE_AT = 1  # Reduced from 2
    cfg.MODEL.FPN.NORM = "GN"

    # --- Modifications for Large Object Detection ---
    # RetinaNet uses 5 feature levels by default, keep those but optimize them
    # Default feature levels are P3, P4, P5, P6, P7
    
    # IMPORTANT: Make sure we have 5 sets of anchor sizes to match the 5 feature levels
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [
        [32, 40.3, 51],           # For P3 (smaller objects)
        [64, 80.6, 102],          # For P4
        [128, 161.3, 203],        # For P5
        [256, 322.5, 406],        # For P6
        [512, 645.0, 812]         # For P7 (larger objects)
    ]

    # Add more aspect ratios for better large object detection
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0, 3.0]]
    
    # Make sure the strides match the feature levels
    cfg.MODEL.ANCHOR_GENERATOR.STRIDES = [8, 16, 32, 64, 128]
    # --- End of Modifications for Large Object Detection ---

    cfg.SOLVER.BASE_LR = 0.002  # Same as original
    cfg.SOLVER.WEIGHT_DECAY = 0.0001
    cfg.SOLVER.MOMENTUM = 0.9
    cfg.SOLVER.WARMUP_FACTOR = 1.0 / 1000
    cfg.SOLVER.WARMUP_ITERS = 100
    cfg.SOLVER.WARMUP_METHOD = "linear"
    cfg.SOLVER.MAX_ITER = 2000  # Keep same as original
    cfg.SOLVER.CHECKPOINT_PERIOD = 500
    cfg.SOLVER.LR_SCHEDULER_NAME = "WarmupCosineLR"  # Changed from constant name

    cfg.TEST.EVAL_PERIOD = 1000
    cfg.OUTPUT_DIR = "/kaggle/working/retinanet_101_large_obj"

    return cfg

# 5. Train + Eval (Minor change in output directory)
def train_enhanced_model():
    cfg = setup_enhanced_model()
    register_datasets()

    trainer = MixedPrecisionTrainer(cfg) #already defined elsewhere
    trainer.resume_or_load(resume=False)
    trainer.train()

    print("\n📊 Evaluating model...")
    evaluator = COCOEvaluator("cottonweed_val", cfg, False, output_dir=cfg.OUTPUT_DIR)
    val_loader = build_detection_test_loader(cfg, "cottonweed_val")
    print(inference_on_dataset(trainer.model, val_loader, evaluator))

    return cfg

# 6. Run Training (No changes needed here)
if __name__ == "__main__":
    train_enhanced_model()

model_final_971ab9.pkl: 228MB [00:00, 271MB/s]                           
2025-04-21 10:19:31.372447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745230771.610688      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745230771.673196      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered



📊 Evaluating model...
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.619
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.807
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.694
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.620
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.486
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.739
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.782
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Recal